<a href="https://colab.research.google.com/github/mrmcgruder98/Assignment-1-PSD/blob/main/Assignment_1_McGruder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)

local_path = Path("data/Assignment_1_PDS.csv")
data_url = "https://raw.githubusercontent.com/mrmcgruder98/Assignment-1-PSD/refs/heads/main/Assignment_1_PDS.csv"
source = local_path if local_path.exists() else data_url

df_raw = pd.read_csv(source)
print(f"Dataset loaded from: {source}")

Dataset loaded from: https://raw.githubusercontent.com/mrmcgruder98/Assignment-1-PSD/refs/heads/main/Assignment_1_PDS.csv


In [ ]:
df_clean = df_raw.copy()
df_clean['Height_m'] = df_clean["Height"] * 0.0254
df_clean['Weight_kg'] = df_clean["Weight"] * 0.453592
df_clean['BMI'] = df_clean['Weight_kg'] / df_clean['Height_m'] ** 2

print(df_clean)


   Height  Weight  Age  Grip Strength  Frailty  Height_m  Weight_kg        BMI
0    65.8     112   30             30        N   1.67132  50.802304  18.187131
1    71.5     136   19             31        N   1.81610  61.688512  18.703582
2    69.4     153   45             29        N   1.76276  69.399576  22.334202
3    68.2     142   22             28        Y   1.73228  64.410064  21.464340
4    67.8     144   29             24        Y   1.72212  65.317248  22.024246
5    68.7     123   50             26        N   1.74498  55.791816  18.322705
6    69.8     141   51             22        Y   1.77292  63.956472  20.347273
7    70.1     136   23             20        Y   1.78054  61.688512  19.458118
8    67.9     112   17             19        N   1.72466  50.802304  17.079550
9    66.8     120   39             31        N   1.69672  54.431040  18.907159


In [ ]:
age_30_or_less = df_clean[df_clean["Age"] <= 30]
age_30_to_45 = df_clean[(df_clean["Age"] >= 30) & (df_clean["Age"] <= 45)]
age_45_to_60 = df_clean[(df_clean["Age"] >= 45) & (df_clean["Age"] <= 60)]
age_60_or_more = df_clean[df_clean["Age"] >= 60]

# Define the bins and labels for age categorization
bins = [0, 30, 45, 60, np.inf]
labels = ['30 or less', '30 to 45', '45 to 60', '60 or more']

age_categories = pd.cut(df_clean["Age"], bins=bins, labels=labels, include_lowest=True)
dummies = pd.get_dummies(age_categories, prefix='Age_Group', dtype=int)
df_clean = df_clean.assign(**dummies)

In [ ]:
df_clean.columns = df_clean.columns.str.strip()

# Explicitly check if 'Frailty' column exists after stripping
if 'Frailty' not in df_clean.columns:
    # If not, search for a column that contains 'Frailty' (case-insensitive)
    found_frailty_col = None
    for col in df_clean.columns:
        if 'frailty' in col.lower():
            found_frailty_col = col
            break

    if found_frailty_col:
        # Rename the found column to 'Frailty' for consistent access
        df_clean = df_clean.rename(columns={found_frailty_col: 'Frailty'})
        print(f"Renamed column '{found_frailty_col}' to 'Frailty'.")
    else:
        # If no 'Frailty'-like column is found, re-raise the error or handle as appropriate
        raise KeyError("Could not find 'Frailty' or any similar column after stripping and searching.")

df_clean['Frailty_Binary'] = df_clean['Frailty'].map({'Y': 1, 'N': 0}).astype('int8')
print(df_clean)

   Height  Weight  Age  Grip Strength Frailty  Height_m  Weight_kg        BMI  \
0    65.8     112   30             30       N   1.67132  50.802304  18.187131   
1    71.5     136   19             31       N   1.81610  61.688512  18.703582   
2    69.4     153   45             29       N   1.76276  69.399576  22.334202   
3    68.2     142   22             28       Y   1.73228  64.410064  21.464340   
4    67.8     144   29             24       Y   1.72212  65.317248  22.024246   
5    68.7     123   50             26       N   1.74498  55.791816  18.322705   
6    69.8     141   51             22       Y   1.77292  63.956472  20.347273   
7    70.1     136   23             20       Y   1.78054  61.688512  19.458118   
8    67.9     112   17             19       N   1.72466  50.802304  17.079550   
9    66.8     120   39             31       N   1.69672  54.431040  18.907159   

   Age_Group_30 or less  Age_Group_30 to 45  Age_Group_45 to 60  \
0                     1                  

In [ ]:
# Create the 'reports' directory if it doesn't exist
reports_dir = Path('reports')
reports_dir.mkdir(exist_ok=True)

# Compute summary statistics for numeric columns (mean, median, std)
numeric_cols = df_clean.select_dtypes(include=np.number).columns
summary_stats = df_clean[numeric_cols].agg(['mean', 'median', 'std']).T

# Save the summary table to a markdown file
with open(reports_dir / 'findings.md', 'w') as f:
    f.write("# Data Summary and Findings\n\n")
    f.write("## Summary Statistics for Numeric Columns\n\n")
    f.write(summary_stats.to_markdown(numalign="left", stralign="left"))
    f.write("\n\n")

print("Summary statistics saved to reports/findings.md")
display(summary_stats)

Summary statistics saved to reports/findings.md


,mean,median,std
Height,68.600000,68.450000,1.670662
Weight,131.900000,136.000000,14.231811
Age,32.500000,29.500000,12.860361
Grip Strength,26.000000,27.000000,4.521553
Height_m,1.742440,1.738630,0.042435
Weight_kg,59.828785,61.688512,6.455436
BMI,19.682831,19.182638,1.782710
Age_Group_30 or less,0.600000,1.000000,0.516398
Age_Group_30 to 45,0.200000,0.000000,0.421637
Age_Group_45 to 60,0.200000,0.000000,0.421637
